In [ ]:
from glob import glob
import os
import xarray as xr
from luccpy import calc


def calc_sif_spei_corr(siffile, speifile, outfile):
    sif = xr.open_dataset(siffile)
    spei = xr.open_dataset(speifile)

    sif = sif.sel(time=(sif["time"].dt.year >= 2007) & (sif["time"].dt.year <= 2016))
    spei = spei.sel(time=(spei["time"].dt.year >= 2007) & (spei["time"].dt.year <= 2016))

    sif_jp = sif - sif.mean(dim="time")
    spei_jp = spei - spei.mean(dim="time")

    spei_jp["time"] = sif_jp.time.values

    spearman_corr = calc.calc_bi_corr_rp(
        spei_jp.spei, sif_jp.SIF, dim="time", method="spearman"
    )

    spearman_corr.to_netcdf(outfile)


clear_filelist = glob(os.path.join("PATH_TO_SIF_INPUT", "*selmon*.nc"))
clear_filelist = sorted(clear_filelist)

for jj in range(1, 49):
    spei_filelist = glob(os.path.join("PATH_TO_SPEI_INPUT", f"spei{jj:02}*selmon*.nc"))
    spei_filelist = sorted(spei_filelist)

    for ii in range(len(clear_filelist)):
        siffile = clear_filelist[ii]
        speifile = spei_filelist[ii]

        outfile = (
            speifile
            .replace("INPUT_SPEI_DIRNAME", "OUTPUT_SPEARMAN_DIRNAME")
            .replace("INPUT_YEAR_TAG", "OUTPUT_CORR_TAG")
        )

        calc_sif_spei_corr(siffile, speifile, outfile)


In [ ]:
# coding:utf-8
import numpy as np
import os
import xarray as xr
from tqdm import tqdm
from datetime import datetime


def read_nc_data(nc_file_path):
    return xr.open_dataset(nc_file_path)


def extract_time_scale_and_month(file_name):
    time_scale_part = file_name.split('spei')[1].split('_spearman_corr')[0]
    time_scale = time_scale_part.split('_')[0]
    month = file_name.split('_selmonth')[1].split('.nc')[0]
    return time_scale, month


def time_series_test(inputpath, outputPath, time_window, season):
    nc_files = [f for f in os.listdir(inputpath) if f.endswith('.nc') and f.startswith('spei')]
    max_values = np.full((360, 720), np.nan)
    time_scales = np.empty((360, 720), dtype=object)
    months = np.empty((360, 720), dtype=object)

    if not nc_files:
        return

    ds = read_nc_data(os.path.join(inputpath, nc_files[0]))
    lats = ds['lat'].values
    lons = ds['lon'].values

    for file_name in tqdm(nc_files):
        ds = read_nc_data(os.path.join(inputpath, file_name))
        time_scale, month = extract_time_scale_and_month(file_name)
        data_var = ds['r'].values

        for i in range(data_var.shape[0]):
            for j in range(data_var.shape[1]):
                if not np.isnan(data_var[i, j]):
                    if np.isnan(max_values[i, j]) or abs(data_var[i, j]) > abs(max_values[i, j]):
                        max_values[i, j] = data_var[i, j]
                        time_scales[i, j] = str(time_scale)
                        months[i, j] = str(month)

    output_filename = os.path.join(outputPath, f"max_corr_{time_window}_{season}.nc")

    max_ds = xr.Dataset({
        'max_correlation': (['lat', 'lon'], max_values),
        'time_scale': (['lat', 'lon'], time_scales),
        'month': (['lat', 'lon'], months)
    }, coords={
        'lat': (('lat'), lats),
        'lon': (('lon'), lons)
    })
    max_ds.to_netcdf(output_filename)


base_input_path = r"PATH_TO_BASE_INPUT"
output_path = r"PATH_TO_OUTPUT"

time_windows = [f"{start}-{end}" for start, end in zip(range(2001, 2012), range(2010, 2021))]
seasons = ["Spring", "Summer", "Autumn", "Winter"]

for time_window in time_windows:
    for season in seasons:
        input_path = os.path.join(
            base_input_path,
            f"OUTPUT_SPEARMAN_DIR_{time_window}",
            f"{time_window}_{season}"
        )
        time_series_test(input_path, output_path, time_window, season)
